# Lecture 5: CTMC, дискретные paths и маленький denoiser

Дополнение к [Lecture 5](README.md), созданное по §7 notes / slides pp.5–28.
Это новая учебная работа, не официальный четвёртый MIT lab.
Все default cells работают на CPU, без downloads и больших весов.

Сначала сравним exact CTMC law с численным и Monte Carlo расчётом. Затем обучим маленький
categorical posterior для двух зависимых tokens. Это ещё не генеративная language model.


In [ ]:
from pathlib import Path
import json, os
import numpy as np
import torch

def find_root():
    for p in (Path.cwd(),*Path.cwd().parents):
        if (p/'curriculum/references/materials_manifest.json').is_file():return p
    raise FileNotFoundError('Open this notebook from generative-ai-research.')
ROOT=find_root();OUT=ROOT/'curriculum/artifacts';OUT.mkdir(exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR',str(OUT/'mpl-cache'))
import matplotlib.pyplot as plt
torch.set_num_threads(4);torch.manual_seed(184)
rng=np.random.default_rng(184)
DEVICE='cpu'  # All default experiments are small and use no downloaded weights.
results={'scope':'small educational CPU experiments; no pretrained inference','mlx':'not_run'}
print('Device:',DEVICE,'Torch:',torch.__version__)


## 1. Rate matrix — не transition probability matrix

Фиксируем row convention: Q[i,j] — rate из i в j; off-diagonal entries неотрицательны,
диагональ равна минус сумме outgoing rates, строки суммируются в 0.
Для row probability vector: $dp/dt=pQ$, $p(t)=p(0)\exp(tQ)$.
Вероятности переходов за малый h: $P(h)\approx I+hQ$; rate может быть больше 1.


In [ ]:
a,b=1.2,.7
Q=torch.tensor([[-a,a],[b,-b]],dtype=torch.float64)
p0=torch.tensor([1.,0.],dtype=torch.float64)
assert torch.allclose(Q.sum(-1),torch.zeros(2,dtype=Q.dtype))
assert (Q[~torch.eye(2,dtype=torch.bool)]>=0).all()
T=2.;exact=p0@torch.matrix_exp(T*Q)
stationary=torch.tensor([b,a],dtype=Q.dtype)/(a+b)
assert torch.allclose(stationary@Q,torch.zeros(2,dtype=Q.dtype),atol=1e-12)
assert torch.allclose(exact.sum(),torch.tensor(1.,dtype=Q.dtype))
ts=torch.linspace(0,4,101,dtype=Q.dtype)
curve=torch.stack([p0@torch.matrix_exp(t*Q) for t in ts])
plt.figure(figsize=(6,3));plt.plot(ts,curve[:,1],label='P(Xt=1) exact')
plt.axhline(float(stationary[1]),ls='--',label='stationary');plt.legend();plt.xlabel('time');plt.tight_layout();plt.savefig(OUT/'ctmc_probability.png');plt.show()
print('Exact at T:',exact,'Stationary:',stationary)


## 2. Ошибка шага и нарушение positivity

При слишком большом h матрица I+hQ имеет отрицательные элементы и не является transition matrix.
Не исправляйте её clipping без объявления другого алгоритма.
Сравним допустимые Euler grids с exact matrix exponential.


In [ ]:
euler_rows=[]
for steps in [1,4,16,64,256]:
    h=T/steps;transition=torch.eye(2,dtype=Q.dtype)+h*Q
    valid=bool((transition>=0).all())
    state=p0@torch.linalg.matrix_power(transition,steps)
    error=float((state-exact).abs().max())
    euler_rows.append({'steps':steps,'h':h,'transition_valid':valid,'max_probability_error':error})
    print(euler_rows[-1])
assert not euler_rows[0]['transition_valid']
valid_rows=[r for r in euler_rows if r['transition_valid']]
assert valid_rows[-1]['max_probability_error']<valid_rows[0]['max_probability_error']
results['finite_step']=euler_rows


## 3. Gillespie: точные времена скачков для постоянного Q

В state i время ожидания имеет Exponential(rate=-Qii); destination распределён пропорционально outgoing rates.
В двух состояниях destination единственный. Exact относится к simulation law при постоянном Q;
finite number of trajectories всё равно даёт Monte Carlo error.


In [ ]:
N=12000
states=np.zeros(N,dtype=np.int64);times=np.zeros(N);jumps=np.zeros(N,dtype=np.int64)
while True:
    active=np.flatnonzero(times<T)
    if not len(active):break
    rates=np.where(states[active]==0,a,b)
    next_times=times[active]+rng.exponential(1/rates)
    take=next_times<T;moving=active[take]
    states[moving]=1-states[moving];jumps[moving]+=1
    times[active]=next_times
empirical=float(states.mean());truth=float(exact[1])
standard_error=np.sqrt(truth*(1-truth)/N)
assert abs(empirical-truth)<5*standard_error+1/N
results['gillespie']={'trajectories':N,'empirical_probability_state_1':empirical,
    'exact_probability_state_1':truth,'binomial_standard_error':standard_error,'total_jumps':int(jumps.sum())}
print(results['gillespie'])
plt.figure(figsize=(5,3));plt.bar(['exact','Monte Carlo'],[truth,empirical],yerr=[0,1.96*standard_error])
plt.ylim(0,1);plt.ylabel('P(XT=1)');plt.tight_layout();plt.show()


## 4. Factorized masking path для зависимых tokens

Данные: с вероятностью 1/2 строка `00`, с вероятностью 1/2 строка `11`.
Marginal каждого token одинакова, но tokens зависимы. Corruption независимо оставляет каждый token
с вероятностью kappa(t)=t, иначе заменяет на MASK=2.

Denoising target — categorical posterior clean token. Для `[MASK,MASK]` он равен [.5,.5];
если увидели хотя бы один clean token, второй однозначно известен.
Обучим MLP с cross-entropy. Независимый corruption не делает data tokens независимыми.
Для полностью masked input правильный posterior остаётся неопределённым: population loss не должен стремиться к нулю.
На фиксированном validation вычислим NLL аналитического Bayes posterior как ориентир, а не будем оценивать успех по нулевому loss.


In [ ]:
MASK=2
class TokenPosterior(torch.nn.Module):
    def __init__(self):
        super().__init__();self.embedding=torch.nn.Embedding(3,8)
        self.net=torch.nn.Sequential(torch.nn.Linear(17,32),torch.nn.SiLU(),torch.nn.Linear(32,4))
    def forward(self,x,t):return self.net(torch.cat([self.embedding(x).flatten(1),t],-1)).reshape(-1,2,2)

def batch(n,generator=None):
    bit=torch.randint(2,(n,1),generator=generator)
    clean=bit.repeat(1,2);t=.05+.9*torch.rand(n,1,generator=generator)
    keep=torch.rand(n,2,generator=generator)<t
    noisy=torch.where(keep,clean,torch.full_like(clean,MASK))
    return noisy,t,clean

model=TokenPosterior();optimizer=torch.optim.Adam(model.parameters(),lr=.02)
val=batch(4096,torch.Generator().manual_seed(900))
oracle_validation_ce=float((val[0]==MASK).all(dim=1).float().mean())*np.log(2)
losses=[];validation=[]
with torch.no_grad():initial=float(torch.nn.functional.cross_entropy(model(val[0],val[1]).reshape(-1,2),val[2].reshape(-1)))
for step in range(140):
    noisy,t,clean=batch(256)
    logits=model(noisy,t)
    loss=torch.nn.functional.cross_entropy(logits.reshape(-1,2),clean.reshape(-1))
    optimizer.zero_grad();loss.backward();optimizer.step();losses.append(float(loss.detach()))
    if step%5==0:
        with torch.no_grad():validation.append((step,float(torch.nn.functional.cross_entropy(model(val[0],val[1]).reshape(-1,2),val[2].reshape(-1)))))
with torch.no_grad():final=float(torch.nn.functional.cross_entropy(model(val[0],val[1]).reshape(-1,2),val[2].reshape(-1)))
assert np.isfinite(final) and final<initial
results['toy_denoising']={'parameters':sum(p.numel() for p in model.parameters()),'steps':140,
 'initial_validation_cross_entropy':initial,'final_validation_cross_entropy':final,
 'bayes_posterior_validation_cross_entropy':oracle_validation_ce,
 'scope':'two-token synthetic distribution only; not natural text or full CTMC generation'}
print(results['toy_denoising'])
plt.figure(figsize=(6,3));plt.plot(losses,alpha=.5,label='train');plt.plot(*zip(*validation),label='validation')
plt.axhline(oracle_validation_ce,ls='--',label='Bayes posterior on validation')
plt.xlabel('training step');plt.ylabel('token cross-entropy');plt.legend();plt.tight_layout();plt.show()


## 5. Сверка с Bayes posterior и построение локальных rates

При interior t outgoing rates для masked token равны posterior/(1-t).
Диагональ устанавливается как минус outgoing sum. Проверка одной локальной rate row не заменяет
sampling полной sequence CTMC по временам.


In [ ]:
cases=torch.tensor([[2,2],[0,2],[2,1],[0,0],[1,1]])
expected=torch.tensor([[.5,.5],[1.,1.],[0.,0.],[1.,1.],[0.,0.]]) # P(clean token = 0), both positions
with torch.no_grad():predicted=model(cases,torch.full((5,1),.5)).softmax(-1)[...,0]
posterior_error=float((predicted-expected).abs().max())
print('Exact P(token=0):',expected);print('Predicted:',predicted)
print('Max posterior error:',posterior_error)
# For masked state at t=.5, destination probabilities are probabilities for 0/1.
prob=torch.tensor([.5,.5],dtype=torch.float64);t=.5
rate_row=torch.cat([prob/(1-t),torch.tensor([-1/(1-t)],dtype=prob.dtype)])
assert rate_row[:2].min()>=0 and torch.allclose(rate_row.sum(),torch.tensor(0.,dtype=prob.dtype))
results['toy_posterior_max_error']=posterior_error
results['full_learned_ctmc_sampling']='not_implemented_in_this_intro_lab'
(OUT/'ctmc-results.json').write_text(json.dumps(results,indent=2)+chr(10))
print('Saved:',OUT/'ctmc-results.json')


## Упражнения и вывод

1. Изменить rates при том же stationary ratio: как меняется скорость mixing?
2. Сравнить Monte Carlo uncertainty при N и 4N. Почему error не обязан падать в каждом отдельном run?
3. Сделать initial distribution произвольной; проверить exact law и Gillespie.
4. Заменить зависимые data tokens независимыми. Что меняется в posterior для частично masked input?
5. Реализовать генерацию sequence CTMC на t<1, отдельно контролируя growing rates возле endpoint.
6. Написать вывод по сохранённым числам: отдельно numerical integration, Monte Carlo и learned posterior errors.

Этот notebook показывает CTMC law и denoising objective. Он не утверждает обучение language model,
не реализует production masked-language sampler и не превращает continuous video DiT в CTMC.
Для ComfyUI можно импортировать сохранённый plot как image; стандартный KSampler не вычисляет этот алгоритм.
